# A Goal-Based Agent for Warehouse Navigation

Notebook for the *Agents* lab. An LLM (Claude) was used to help write the path-finding code,
following the lab's own suggested prompt; the results below all come from running this
notebook end to end.

## Task 1 - Understanding the problem

1. **Environment:** a fixed 7x21 grid warehouse with shelving (`#`) that cannot be crossed.
   It is static, fully observable and deterministic.
2. **Goal:** reach the cell marked `G`, starting from `S`, without ever entering a `#` cell.
3. **Available actions:** Up, Down, Left, Right - each shifts the vehicle by one grid cell.
4. **What must be maintained to choose the next action:** the vehicle's current position,
   the full map, the goal location, and a record of which cells have already been visited
   (otherwise the search could revisit the same cell forever).
5. **Why goal-based rather than simple reflex:** a simple reflex agent reacts to the
   immediate percept with a fixed rule ("if there's a wall ahead, turn") and has no way to
   recognise a dead end before walking into it. This agent instead searches ahead over
   possible action sequences and only commits to one once it is known to reach the goal.

> **Think about it.** Doubling the warehouse's size would not break breadth-first
> search in principle, but it would roughly double the number of cells and therefore grow the
> number of cells explored substantially - and if several vehicles or moving obstacles were
> introduced, a plan computed once would need to be rechecked or replanned as the
> environment changed, which a single static-map search does not handle.

## Task 2 - Designing the agent

| Component | Role in this problem |
|---|---|
| Environment | the warehouse grid |
| Current state | the vehicle's `(row, col)` position |
| Goal | the coordinates of `G` |
| Actions | Up, Down, Left, Right |
| Decision procedure | breadth-first search over reachable positions |

```
map + current position -> [search: expand, test against goal] -> sequence of actions -> executed one at a time
```

In [1]:
WAREHOUSE = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################",
]

for row in WAREHOUSE:
    print(row)
print(f"\n{len(WAREHOUSE)} rows x {len(WAREHOUSE[0])} columns")

#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################

7 rows x 21 columns


## Task 3 - Prompt engineering and the generated agent

Prompt used, based on the lab's suggested wording:

> *Write a well-documented Python program implementing a goal-based agent for this warehouse
> navigation problem. Represent the warehouse as a 2D grid, find a collision-free path from
> S to G, avoid all obstacles, print the path or report that none exists, and explain which
> search algorithm was chosen and why.*

**Algorithm chosen: breadth-first search.** Every move has the same cost (1), so BFS is
guaranteed to return a shortest path, needs no heuristic to design, and is the simplest
correct choice for this grid.

In [2]:
from collections import deque

MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}

def find_cell(grid, symbol):
    for r, row in enumerate(grid):
        c = row.find(symbol)
        if c != -1:
            return (r, c)
    raise ValueError(f"{symbol!r} not found on the map")

def legal_moves(grid, pos):
    r, c = pos
    for action, (dr, dc) in MOVES.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[nr]) and grid[nr][nc] != "#":
            yield action, (nr, nc)

In [3]:
def find_path(grid):
    start, goal = find_cell(grid, "S"), find_cell(grid, "G")
    frontier = deque([start])
    came_from = {start: None}

    while frontier:
        current = frontier.popleft()
        if current == goal:
            break
        for action, nxt in legal_moves(grid, current):
            if nxt not in came_from:
                came_from[nxt] = (current, action)
                frontier.append(nxt)

    if goal not in came_from:
        return None

    actions, cur = [], goal
    while came_from[cur] is not None:
        cur, action = came_from[cur]
        actions.append(action)
    actions.reverse()
    return actions

In [4]:
def draw(grid, actions):
    marked = [list(row) for row in grid]
    r, c = find_cell(grid, "S")
    for a in actions[:-1]:
        dr, dc = MOVES[a]
        r, c = r + dr, c + dc
        marked[r][c] = "*"
    return "\n".join("".join(row) for row in marked)

path = find_path(WAREHOUSE)
if path is None:
    print("No path exists from S to G.")
else:
    print(f"Path found, {len(path)} moves: {', '.join(path)}")
    print()
    print(draw(WAREHOUSE, path))

Path found, 20 moves: Right, Right, Right, Down, Right, Right, Right, Up, Right, Right, Right, Right, Right, Right, Right, Right, Right, Right, Right, Right

#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


A 20-move path is found. That is a reasonable length given that `S` and `G` are 18
columns apart on the same row, with a short detour needed around the wall segment just past
column 5.

### Checking the generated program before trusting it

An LLM-written program that runs is not automatically correct, so three independent checks
are run: replaying the returned path never crosses a wall and ends on `G`; a map where `G` is
sealed off reports no path; and a tiny map with `G` next to `S` gives a one-move path.

In [5]:
def replay(grid, actions):
    r, c = find_cell(grid, "S")
    for a in actions:
        dr, dc = MOVES[a]
        r, c = r + dr, c + dc
        assert grid[r][c] != "#", "replayed path crosses an obstacle"
    return (r, c) == find_cell(grid, "G")

assert replay(WAREHOUSE, path)
print("Test 1 passed: the path stays on free cells and ends exactly at G")

sealed_off = ["#######",
              "#S.#.G#",
              "#######"]
assert find_path(sealed_off) is None
print("Test 2 passed: a sealed-off goal correctly reports no path")

adjacent = ["#####",
            "#SG##",
            "#####"]
assert find_path(adjacent) == ["Right"]
print("Test 3 passed: an adjacent goal gives a single-move path")

Test 1 passed: the path stays on free cells and ends exactly at G
Test 2 passed: a sealed-off goal correctly reports no path
Test 3 passed: an adjacent goal gives a single-move path


**Questions**

1. **Did the LLM generate a working program on the first attempt?** Yes - it ran without
   errors and returned the 20-move path above.
2. **How could the prompt be improved?** By asking explicitly for a visual rendering of the
   path and for a test case where no path exists, since running it once on the given map
   alone never exercises the "no solution" branch.
3. **Which search algorithm did the LLM choose?** Breadth-first search.
4. **Why this algorithm?** Because every move costs the same, BFS is guaranteed to return a
   shortest path without needing a heuristic. It's worth noting that "the most efficient
   algorithm" is not automatically A\* here - A\* only beats BFS when its heuristic lets it
   prune real branching in the state space, and on a narrow corridor-like map (as the search
   lab's comparison shows) it can end up expanding exactly as many states as BFS.